In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

RAW_DIR = "/Workspace/Users/yashikumawat53@gmail.com/Drafts/drone_pipeline/data/raw"
BRONZE_DIR = "/Volumes/workspace/default/my_volume/drone_pipeline/data/bronze"

In [0]:

def get_spark():
    return (
        SparkSession.builder.appName("DroneBronzeLayer")
        .getOrCreate()
    )

In [0]:
def ingest_csv_to_bronze(spark, file_name, table_name):
    df = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_DIR}/{file_name}")

    bronze_df = (
        df.withColumn("ingestion_time", current_timestamp())
        .withColumn("source_file", lit(file_name))
    )

    out_path = f"{BRONZE_DIR}/{table_name}"
    bronze_df.write.format("parquet").mode("overwrite").save(out_path)

    print(f"[BRONZE] {file_name} -> {out_path}  ({bronze_df.count()} rows)")
    return bronze_df


In [0]:
def main():
    spark = get_spark()
    try:
        spark.sparkContext.setLogLevel("ERROR")
    except Exception:
        pass
    ingest_csv_to_bronze(spark, "drones.csv", "drones")
    ingest_csv_to_bronze(spark, "deliveries.csv", "deliveries")
    ingest_csv_to_bronze(spark, "flight_logs.csv", "flight_logs")

    print("\nBronze layer complete. Raw data is now safely stored in Delta format.")
    spark.stop()

if __name__ == "__main__":
    main()

[BRONZE] drones.csv -> /Volumes/workspace/default/my_volume/drone_pipeline/data/bronze/drones  (25 rows)
[BRONZE] deliveries.csv -> /Volumes/workspace/default/my_volume/drone_pipeline/data/bronze/deliveries  (600 rows)
[BRONZE] flight_logs.csv -> /Volumes/workspace/default/my_volume/drone_pipeline/data/bronze/flight_logs  (600 rows)

Bronze layer complete. Raw data is now safely stored in Delta format.
